# Fase 4 - Diagnóstico 

Este script sirve como revisión de los datos  y mide cómo se fragmenta la vecindad con distintas distancias máximas entre vecinos $d_{\text{max}}$ 

Está construido para responder un grupo de preguntas:

|Sección|Pregunta|¿por qué?|
|---|---|---|
|Datos|¿Cada corona tiene sus siete variables? |SKATER no admite valores vacíos|
| Coronas coincidentes| ¿Hay coronas casi en el mismo punto?|La triangulación de Delaunay no puede usar dos puntos idénticos|
|Vecino más cercano|¿A qué distancia está la corona más próxima de cada evento?|Muestra si un umbral de distancia deja eventos sin vecinos|
|Grafo|¿Cuántos grupos sueltos quedan con cada $d_{\text{max}}$ |Un grafo partido en muchos grupos no deja formar regiones grandes|
|Correlaciones|¿Hay pares de variables casi iguales?|SKATER usa las siete con igual peso; dos variables muy correlacionadas contarían casi el doble|
|Valores extremos|¿Algún evento se aparta mucho en alguna variable?|Un valor extremo pesa mucho en la suma de cuadrados que minimiza SKATER|


**Datos de entrada**

Las entradas de este script son los resultados de los scripts de la fase 3: coronas e identificadores.
Las tablas con las siete variables evaluadas por cada evento y una capa del río Medellín.


**SALIDAS**

En este caso no se genera ningún archivo de salida. Todos los resultados se imprimen en la última celda como un reporte de diagnóstico. 



In [8]:
import arcpy   #lectura de capas en ArcGIS Pro

# Manejo de coordenadas y tablas
import numpy as np
import pandas as pd


from scipy.spatial import Delaunay, cKDTree  # Busqueda de vecinos cercanos y triangulación de las coronas

# Matriz de conexiones del grafo y cuentan cuántos grupos conectados tiene
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

## Configuración 

### **¿Qué hace?**


Reúne las rutas, los campos y los umbrales que se van a probar. Es el bloque editable 

In [18]:

CORONAS = r"D:\Users\tasal\Desktop\Tesis de grado\fase3\F3_fase3_integrado.gpkg\main.fase3_eventos"
CAMPO_ID = "ID"

# Puede ser un .csv o una capa/tabla de ArcGIS; se une a las coronas por el ID.
TABLA_VARIABLES = r"D:/Users/tasal/Desktop/Tesis de grado/fase3/F3_S2_tabla_modelo.csv"
CAMPO_ID_TABLA = "\ufeffID"
VARIABLES = {
    "log_area":      "LOG_AREA",
    "log_HL":        "LOG_HL",
    "log_elong":     "LOG_ELONG",
    "pendiente":     "PENDIENTE",
    "dist_fallas":   "LOG_DIST_ESTRUCTURA",
    "dist_drenajes": "LOG_DIST_DRENAJE",
    "dens_lineam":   "DENS_LIN_1000",
}
RIO = "Rio_Medellin"  # linea del eje del rio

# Candidatos de d_max anclados en la Fase 2:
#   650 m  = fin del ascenso de L(r) - r (tamaño de los cúmulos)
#   1000 m = alcance de los variogramas de marcas de área y H/L (vecinos parecidos)
#   1500, 2000 y 3000 m = referencias hasta el límite de interpretación
DMAX_CANDIDATOS = [650, 1000, 1500, 2000, 3000]

TOL_COINCIDENCIA = 2.0    # un píxel del DEM
TAM_MIN_REGION = 20    # Tamaño mínimo de cada región del SKATER

## Diagnóstico 

### **¿Que hace?**

Realiza la revisión completa. Primero define tres funciones y después imprime las seis secciones del reporte.


|Función|Tarea|
|---|---|
|`clave(v)`|pasa el identificador a texto comparable|
|`leer_tabla(ruta)`|lee la tabla de variables|
|`resumen(mascara, etiqueta)`|arma el grafo con las aristas que deja pasar `mascara` y cuenta sus componentes conectadas, es decir, los grupos de eventos unidos entre sí. Imprime cuántos hay, cuántos eventos quedan aislados, cuántos grupos tienen menos de 20 eventos y el tamaño de los cuatro mayores.|

**Paso a paso**

El flujo de este bloque es el siguiente:

1. Comprueba que la vatriable `CORONAS` sea una capa de puntos; si la ruta apunta al archivo GeoPackage y no a una capa, lista las capas que tiene. Lee identificadores y coordenadas, lee la tabla, convierte las variables a números y las ordena como las coronas. Reporta identificadores repetidos, eventos sin fila en la tabla y vacíos.

2.  Lista los pares de coronas a menos de 2 m.

3.  busca los dos puntos más cercanos a cada corona e imprime los percentiles

4.  Triangula las coronas (Delaunay), saca las aristas de los triángulos sin repetirlas y mide su largo. Une los tramos del río y marca las aristas cuyo segmento toca la línea.

5.   Calcula la correlación de Pearson entre las siete variables y marca los pares con $\lvert x \rvert> 0,7$

6.   Estandariza cada variable ( 
z
  es el valor menos la media, dividido por la desviación) y lista los casos con $\lvert z \rvert> 4$ 


In [19]:
def clave(v):
    """ID como texto comparable (evita que 73, 73.0 y '73' no coincidan)."""
    try:
        return str(int(float(v)))
    except (TypeError, ValueError):
        return str(v).strip()
 
 
def leer_tabla(ruta):
    if ruta.lower().endswith(".csv"):
        for cod in ("utf-8", "latin-1"):
            try:
                return pd.read_csv(ruta, sep=None, engine="python", encoding=cod)
            except UnicodeDecodeError:
                continue
        raise SystemExit("No se pudo leer el CSV (codificacion).")
    campos_t = [f.name for f in arcpy.ListFields(ruta) if f.type not in ("Geometry", "Blob", "Raster")]
    return pd.DataFrame(list(arcpy.da.SearchCursor(ruta, campos_t)), columns=campos_t)
 
 
print("=" * 70)
print("1. DATOS")
desc = arcpy.Describe(CORONAS)
if not hasattr(desc, "spatialReference"):
    msg = (f"CORONAS no apunta a una capa de puntos: ArcGIS la ve como '{desc.dataType}'.\n"
           f"Ruta usada: {CORONAS}")
    if desc.dataType == "Workspace":
        arcpy.env.workspace = CORONAS
        msg += f"\nCapas dentro de ese archivo: {arcpy.ListFeatureClasses()}"
    raise SystemExit(msg)
sr = desc.spatialReference
print(f"Sistema de coordenadas: {sr.name} | unidad: {sr.linearUnitName}")
filas = list(arcpy.da.SearchCursor(CORONAS, [CAMPO_ID, "SHAPE@XY"]))
ids = np.array([f[0] for f in filas])
xy = np.array([f[1] for f in filas], dtype=float)
n = len(ids)
 
tabla = leer_tabla(TABLA_VARIABLES)
faltan = [c for c in [CAMPO_ID_TABLA] + list(VARIABLES.values()) if c not in tabla.columns]
if faltan:
    raise SystemExit(f"Faltan campos en la tabla: {faltan}\nCampos de la tabla: {list(tabla.columns)}")
for c in VARIABLES.values():
    if not pd.api.types.is_numeric_dtype(tabla[c]):
        tabla[c] = pd.to_numeric(tabla[c].astype(str).str.replace(",", "."), errors="coerce")
tabla.index = [clave(v) for v in tabla[CAMPO_ID_TABLA]]
tabla = tabla[~tabla.index.duplicated()]
claves = [clave(v) for v in ids]
sin_datos = [k for k in claves if k not in tabla.index]
X = tabla.reindex(claves)[list(VARIABLES.values())].to_numpy(dtype=float)
nombres = list(VARIABLES)
 
print(f"Eventos en la capa: {n} | IDs repetidos: {n - len(set(claves))} | "
      f"filas en la tabla: {len(tabla)} | eventos sin fila en la tabla: {len(sin_datos)}")
if sin_datos:
    print(f"  ATENCION IDs sin datos: {sin_datos[:20]}")
nulos = np.isnan(X).sum(axis=0)
if nulos.sum() == 0:
    print("Sin vacios en las 7 variables.")
for nombre, k in zip(nombres, nulos):
    if k:
        print(f"  ATENCION {nombre}: {int(k)} vacios")
 
print("\n2. CORONAS COINCIDENTES")
arbol = cKDTree(xy)
pares = sorted(arbol.query_pairs(TOL_COINCIDENCIA))
print(f"Pares a menos de {TOL_COINCIDENCIA} m: {len(pares)}")
for a, b in pares:
    print(f"  {ids[a]} - {ids[b]}  ({np.linalg.norm(xy[a] - xy[b]):.1f} m)")
 
print("\n3. DISTANCIA AL VECINO MAS CERCANO (m)")
d_vmc = arbol.query(xy, k=2)[0][:, 1]
for p in (0, 25, 50, 75, 90, 95, 100):
    print(f"  P{p:>3}: {np.percentile(d_vmc, p):8.0f}")
 
print("\n4. GRAFO: DELAUNAY, FILTRO DEL RIO Y d_max")
tri = Delaunay(xy)
aristas = set()
for s in tri.simplices:
    for a, b in ((s[0], s[1]), (s[1], s[2]), (s[0], s[2])):
        aristas.add((min(a, b), max(a, b)))
aristas = np.array(sorted(aristas))
largo = np.linalg.norm(xy[aristas[:, 0]] - xy[aristas[:, 1]], axis=1)
fuera = sorted(set(range(n)) - set(aristas.ravel().tolist()))
print(f"Aristas de Delaunay: {len(aristas)} | eventos fuera de la triangulacion: {len(fuera)}")
 
rio = None
with arcpy.da.SearchCursor(RIO, ["SHAPE@"]) as cur:
    for (g,) in cur:
        rio = g if rio is None else rio.union(g)
if rio.spatialReference.name != sr.name:
    rio = rio.projectAs(sr)
cruza = np.array([
    not arcpy.Polyline(arcpy.Array([arcpy.Point(*xy[a]), arcpy.Point(*xy[b])]), sr).disjoint(rio)
    for a, b in aristas
])
print(f"Aristas que cruzan el rio (se eliminan): {int(cruza.sum())}")
 
 
def resumen(mascara, etiqueta):
    e = aristas[mascara]
    A = coo_matrix((np.ones(len(e)), (e[:, 0], e[:, 1])), shape=(n, n))
    ncomp, comp = connected_components(A, directed=False)
    tam = np.sort(np.bincount(comp))[::-1]
    peq = tam[tam < TAM_MIN_REGION]
    print(f"  {etiqueta:>14} | aristas {len(e):5d} | componentes {ncomp:3d} | "
          f"aislados {int((tam == 1).sum()):3d} | comp.<{TAM_MIN_REGION}: {len(peq):3d} "
          f"({int(peq.sum())} ev.) | mayores: {tam[:4].tolist()}")
 
 
resumen(np.ones(len(aristas), bool), "sin filtros")
resumen(~cruza, "solo rio")
for d in DMAX_CANDIDATOS:
    resumen(~cruza & (largo <= d), f"rio + {d} m")
 
print("\n5. CORRELACIONES ENTRE LAS 7 VARIABLES (se marcan |r| > 0,7)")
ok = ~np.isnan(X).any(axis=1)
R = np.corrcoef(X[ok].T)
for i in range(len(nombres)):
    for j in range(i + 1, len(nombres)):
        if abs(R[i, j]) > 0.7:
            print(f"  ATENCION {nombres[i]} - {nombres[j]}: r = {R[i, j]:.2f}")
print(f"  |r| maximo: {np.max(np.abs(R[np.triu_indices(len(nombres), 1)])):.2f}")
 
print("\n6. VALORES EXTREMOS (|z| > 4)")
Z = (X - np.nanmean(X, axis=0)) / np.nanstd(X, axis=0, ddof=1)
fi, co = np.where(np.abs(np.nan_to_num(Z)) > 4)
if len(fi) == 0:
    print("  Ninguno.")
for i, j in zip(fi, co):
    print(f"  ID {ids[i]} | {nombres[j]} | z = {Z[i, j]:.1f}")
print("=" * 70)
 

1. DATOS
Sistema de coordenadas: MAGNA-SIRGAS_2018_Origen-Nacional | unidad: Meter
Eventos en la capa: 430 | IDs repetidos: 0 | filas en la tabla: 445 | eventos sin fila en la tabla: 0
Sin vacios en las 7 variables.

2. CORONAS COINCIDENTES
Pares a menos de 2.0 m: 1
  425 - 424  (2.0 m)

3. DISTANCIA AL VECINO MAS CERCANO (m)
  P  0:        2
  P 25:       52
  P 50:      115
  P 75:      416
  P 90:      946
  P 95:     1512
  P100:     5357

4. GRAFO: DELAUNAY, FILTRO DEL RIO Y d_max
Aristas de Delaunay: 1275 | eventos fuera de la triangulacion: 0
Aristas que cruzan el rio (se eliminan): 57
     sin filtros | aristas  1275 | componentes   1 | aislados   0 | comp.<20:   0 (0 ev.) | mayores: [430]
        solo rio | aristas  1218 | componentes   1 | aislados   0 | comp.<20:   0 (0 ev.) | mayores: [430]
     rio + 650 m | aristas   555 | componentes 144 | aislados  74 | comp.<20: 140 (335 ev.) | mayores: [26, 25, 23, 21]
    rio + 1000 m | aristas   674 | componentes  98 | aislados  39 